In [ ]:
import json
import os
import random
import re
import shutil
import traceback
from pathlib import Path
from typing import TypedDict


class Instance(TypedDict):
    bbox: tuple[float, float, float, float] | None
    cls_idx: int | None
    keypoints: list[tuple[float, float, int] | None]


def normalize_coords(x: int, y: int, w: int, h: int) -> tuple[float, float]:
    """归一化坐标"""
    return x / w, y / h


def get_name(idx: int, label: str, desc: str) -> str:
    if desc:
        return desc.split("_")[0] if "_" in desc else desc
    elif label in keypoint_map:
        # 只有self有关键点
        return "self"
    elif label == "self":
        return "self"
    else:
        return f"{label}{idx}"


def get_instance_id(desc: str) -> str | None:
    """从description中获取实例id"""
    match = re.search(r"\d+", desc)
    return match.group() if match else None


def labelme2yolo(
    json_path: str,
) -> str:
    """将labelme格式的标注文件转换为yolo格式"""
    with open(json_path, "r") as f:
        data = json.load(f)
    img_path = data["imagePath"]
    img_w = data["imageWidth"]
    img_h = data["imageHeight"]

    instances = {}
    for i, shape in enumerate(data["shapes"]):
        label = shape["label"]
        points = shape["points"]
        desc = shape["description"] or ""

        name = get_name(idx=i, label=label, desc=desc)
        if not name:
            print(
                f"Skipping {name} from description: {desc}, shape: {shape}, json: {json_path}"
            )
        if name not in instances:
            instances[name] = Instance(
                bbox=None, cls_idx=None, keypoints=[None] * len(keypoint_map)
            )

        # 处理矩形框
        if shape["shape_type"] == "rectangle" and label in class_map:
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            shp_w = x_max - x_min
            shp_h = y_max - y_min

            norm_x_center, norm_y_center = normalize_coords(
                x_center, y_center, img_w, img_h
            )
            norm_w, norm_h = normalize_coords(shp_w, shp_h, img_w, img_h)
            instances[name]["bbox"] = (norm_x_center, norm_y_center, norm_w, norm_h)
            instances[name]["cls_idx"] = class_map[label]

        # 处理关键点
        elif shape["shape_type"] == "point" and label in keypoint_map:
            kp_index = keypoint_map[label]
            x, y = points[0]
            norm_x, norm_y = normalize_coords(x, y, img_w, img_h)
            instances[name]["keypoints"][kp_index] = (norm_x, norm_y, 2)

    # 由instances生成yolo格式文本
    yolo_lines = []
    for name, inst in instances.items():
        if inst["bbox"] is None or inst["cls_idx"] is None:
            print(f"Skipping instance: {name} missing bbox or cls_idx from {json_path}")
            continue

        line = [str(inst["cls_idx"])]
        line.extend([f"{coord:6f}" for coord in inst["bbox"]])
        # 顺序补充关键点
        for kp_idx in keypoint_map.values():
            kp = inst["keypoints"][kp_idx]
            if kp is None:
                # 关键点标记为不存在
                line.extend(["0.0", "0.0", "0"])
            else:
                # 关键点标记为可见
                line.extend([f"{kp[0]:6f}", f"{kp[1]:6f}", str(kp[2])])

        yolo_lines.append(" ".join(line))

    # 保存
    filename = os.path.splitext(img_path)[0] + ".txt"
    output = os.path.join(os.path.dirname(json_path), filename)
    with open(output, "w") as f:
        f.write("\n".join(yolo_lines))
    print(f"Successfully converted {json_path} to {output}")

    return output


def make_dataset(labelme_folder: str, dataset_root: str, val_ratio=0.2, seed=None):
    if seed is not None:
        random.seed(seed)
    image_files = [
        f
        for f in os.listdir(labelme_folder)
        if f.lower().endswith(
            (
                ".jpg",
                ".png",
                ".jpeg",
            )
        )
    ]
    json_files = [f for f in os.listdir(labelme_folder) if f.lower().endswith(".json")]
    image_names = {Path(f).stem for f in image_files}
    json_names = {Path(f).stem for f in json_files}

    if image_names != json_names:
        missing_jsons = image_names - json_names
        missing_images = json_names - image_names
        if missing_jsons:
            print(f"Missing JSON files: {missing_jsons}")
        if missing_images:
            print(f"Missing image files: {missing_images}")
        common_names = image_names & json_names
    else:
        common_names = image_names

    all_names = list(common_names)
    random.shuffle(all_names)
    num_val = int(len(all_names) * val_ratio)
    val_names = set(all_names[:num_val])
    train_names = set(all_names[num_val:])

    print(f"Total images: {len(all_names)}")
    print(f"Train images: {len(train_names)}")
    print(f"Val images: {len(val_names)}")

    split_dirs = {
        "train_images": os.path.join(dataset_root, "images", "train"),
        "train_labels": os.path.join(dataset_root, "labels", "train"),
        "val_images": os.path.join(dataset_root, "images", "val"),
        "val_labels": os.path.join(dataset_root, "labels", "val"),
    }
    for dir_path in split_dirs.values():
        os.makedirs(dir_path, exist_ok=True)

    for name in all_names:
        src_img = os.path.join(
            labelme_folder, next(f for f in image_files if Path(f).stem == name)
        )
        src_json = os.path.join(
            labelme_folder, next(f for f in json_files if Path(f).stem == name)
        )
        src_lbl = labelme2yolo(src_json)

        if name in train_names:
            dst_img = os.path.join(
                split_dirs["train_images"], os.path.basename(src_img)
            )
            dst_lbl = os.path.join(
                split_dirs["train_labels"], os.path.basename(src_lbl)
            )
        elif name in val_names:
            dst_img = os.path.join(split_dirs["val_images"], os.path.basename(src_img))
            dst_lbl = os.path.join(split_dirs["val_labels"], os.path.basename(src_lbl))

        shutil.copy2(src_img, dst_img)
        shutil.copy2(src_lbl, dst_lbl)

    print(f"Done. Copied {len(all_names)} files.")


# ================================= 配置和运行 =================================
label_folders = [
    "./dataset/images_warships1",
    "./dataset/images_warships2",
]
label_folder = "./dataset/images_warship"
dataset_root = "./dataset/dataset_warship"
val_ratio = 0.1
random_seed = 42

class_map = {"enemy": 0, "friend": 1}
keypoint_map = {"body": 0, "bow": 1, "stern": 2, "funnel": 3, "smoke": 4}

yaml_name = "data_warship.yaml"

yaml_content = f"""# {yaml_name}
train: {os.path.join(os.path.abspath(dataset_root), "images", "train")}
val: {os.path.join(os.path.abspath(dataset_root), "images", "val")}

nc: 2

names: ['enemy', 'friend']

kpt_shape: [5, 3]  # 5个关键点, 每个关键点有3个坐标
labels: ['body', 'bow', 'stern', 'funnel', 'smoke']
flip_idx: []
"""

if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)
    os.mkdir(dataset_root)

if os.path.exists(label_folder):
    shutil.rmtree(label_folder)
    os.mkdir(label_folder)
for folder in label_folders:
    if not os.path.exists(folder):
        continue
    shutil.copytree(folder, label_folder, dirs_exist_ok=True)

try:
    make_dataset(folder, dataset_root, val_ratio, random_seed)
    yaml_path = os.path.join(Path(dataset_root).parent, yaml_name)
    with open(yaml_path, "w", encoding="utf-8") as yaml_file:
        yaml_file.write(yaml_content)
    print(f"Successfully created dataset")
except Exception as e:
    traceback.print_exc()

shutil.rmtree(label_folder)

In [ ]:
import torch

# 检查 CUDA 是否可用
print("CUDA 可用:", torch.cuda.is_available())

# 检查 PyTorch 是否能识别到您的 GPU
print("GPU 数量:", torch.cuda.device_count())

# 打印 GPU 名称
if torch.cuda.is_available():
    print("GPU 名称:", torch.cuda.get_device_name(0))

# 检查 PyTorch 使用的 CUDA 版本
print("PyTorch CUDA 版本:", torch.version.cuda)

In [ ]:
level = "11s"  # n, s, m, l, x
model_name = f"minimap_{level}"
print(model_name)

In [ ]:
import os
import shutil

import gc
import torch
from ultralytics import YOLO

if os.path.exists(f"runs/pose/{model_name}"):
    shutil.rmtree(f"runs/pose/{model_name}")

# --- 1. 加载模型 ---
# 加载预训练的Pose模型（推荐用于迁移学习）
model = YOLO(f"yolo{level}-pose.pt")
data = r"./dataset/data_warship.yaml"

# --- 2. 训练模型 ---
# 使用您准备好的 data 文件进行训练
try:
    results = model.train(
    data=data,                  # 指向您的数据配置文件
    epochs=100,                 # 训练轮数
    time=20.0,                  # 训练时长（小时）
    patience=300,               # 早停阈值 连续300个epoch没有提升则停止训练
    imgsz=640,                  # 输入图像大小
    batch=-1,                   # 批次大小 (根据GPU显存调整)
    name=f'{model_name}',       # 本次训练的名称
    workers=2,
    # half=True,                  # 半精度训练
    augment=True,               # 启用增强（默认开启）
    # hsv_h=0.015,                # 色调抖动
    # hsv_s=0.7,                  # 饱和度抖动（增强颜色鲁棒性）
    # hsv_v=0.4,                  # 亮度抖动
    # degrees=10,                 # 旋转
    # translate=0.1,              # 平移
    # scale=0.5,                  # 缩放
    # shear=2.0,                  # 剪切
    # flipud=0.0,                 # 上下翻转
    # fliplr=0.5,                 # 左右翻转
    # mosaic=1.0,                 # Mosaic 增强（4图拼接，提升小目标检测）
    # mixup=0.1,                  # MixUp：混合两张图（对抗过拟合）
    # copy_paste=0.2,             # Copy-Paste：将目标粘贴到新位置（模拟密集场景）
)
except Exception as e:
    print(e)
finally:
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
from ultralytics import YOLO
model = YOLO(f"./runs/pose/{model_name}/weights/best.pt")

res = model.export(format="onnx")
print(res)

In [ ]:

# --- 3. 验证模型 ---
# 在验证集上评估训练好的模型
metrics = model.val(data=data)
print("mAP50-95:", metrics.box.map)  # 打印mAP指标


In [ ]:
import cv2
from glob import glob

# --- 4. 预测/推理 ---
# 对单张图片或一个文件夹进行预测

# 加载训练好的最佳权重

model = YOLO(f"./runs/pose/{model_name}/weights/best.pt")  # 路径指向您训练出的最佳模型

# 进行预测
for img in glob(r"C:\Users\Bar\AppData\Roaming\PotPlayerMini64\Capture\*.jpg"):
    results = model.predict(
        source=img,
        conf=0.3,       # 置信度阈值（低置信度=可能是噪声）
        iou=0.45,       # NMS IOU 阈值
        classes=None,   # 检测所有类,
        show=True,      # 会弹窗显示结果
    )
    # --- 5. 处理预测结果 ---
    # results 是一个包含多个结果的列表 (每张图一个结果)
    for result in results:
        # 获取带有关键点的原始图像 (numpy array)
        annotated_frame = result.plot(kpt_radius=5)  # kpt_radius 控制关键点半径

        # 获取关键点数据 (numpy array)
        # 形状: (num_instances, num_keypoints, 3) -> [x, y, visibility]
        keypoints = result.keypoints.data.cpu().numpy()
        print("关键点坐标:", keypoints)

        # 获取边界框数据
        boxes = result.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
        classes = result.boxes.cls.cpu().numpy()  # 类别索引

        # 获取置信度
        confidences = result.boxes.conf.cpu().numpy()

        # 在这里，您可以根据关键点坐标计算航向、速度等
        # 例如，计算船头(bow)到船尾(stern)的向量
        # 注意: 您需要知道关键点在数组中的索引顺序 (由您的data.yaml定义)

        # 显示或保存结果
        cv2.imshow("Result", annotated_frame)
        cv2.waitKey(0)

In [ ]:

# --- 6. 导出模型 (用于部署) ---
# 将模型导出为 ONNX 格式，便于在其他平台运行
success = model.export(format="onnx")